In [30]:
import duckdb

con = duckdb.connect("../data/docintel.duckdb", read_only=True)
con.sql("SHOW contracts").show()
con.sql("SELECT count(*) AS n_contracts FROM contracts").show()
con.sql("SHOW labels").show()
con.sql("SELECT count(*) AS n_labels FROM labels").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ doc_id        │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ title         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ contract_type │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ n_pages       │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ n_chars       │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ txt_path      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pdf_path      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────┐
│ n_contracts │
│    int64    │
├─────────────┤
│         510 │
└─────────────┘

┌─────────────┬─────────────┬─────────

In [13]:
con.sql("""
SELECT contract_type, count(*) AS n
FROM contracts
WHERE contract_type IN ('Supply', 'Franchise', 'Reseller')
GROUP BY contract_type
ORDER BY n DESC
""")

┌───────────────┬───────┐
│ contract_type │   n   │
│    varchar    │ int64 │
├───────────────┼───────┤
│ Supply        │    18 │
│ Franchise     │    15 │
│ Reseller      │    12 │
└───────────────┴───────┘

In [14]:
con.sql("""
SELECT contract_type, count(*) AS n
FROM contracts
GROUP BY contract_type
ORDER BY n DESC, contract_type
""")

┌─────────────────────────┬───────┐
│      contract_type      │   n   │
│         varchar         │ int64 │
├─────────────────────────┼───────┤
│ Maintenance             │    34 │
│ License_Agreements      │    33 │
│ Distributor             │    32 │
│ Strategic Alliance      │    32 │
│ Sponsorship             │    31 │
│ Development             │    29 │
│ Service                 │    28 │
│ Collaboration           │    26 │
│ Co_Branding             │    22 │
│ Hosting                 │    20 │
│    ·                    │     · │
│    ·                    │     · │
│    ·                    │     · │
│ Agency Agreements       │    13 │
│ Transportation          │    13 │
│ Promotion               │    12 │
│ Reseller                │    12 │
│ Consulting Agreements   │    11 │
│ Affiliate_Agreements    │     9 │
│ Endorsement Agreement   │     9 │
│ Joint Venture           │     9 │
│ Non_Compete_Non_Solicit │     3 │
│ Affiliate Agreement     │     1 │
└─────────────────────────┴─

In [15]:
con.sql("""
SELECT contract_type, split_part(pdf_path, '/', 5) AS part, count(*) AS n
FROM contracts
WHERE contract_type LIKE 'Affiliate%'
   OR contract_type LIKE 'Endorsement%'
   OR contract_type LIKE 'Joint Venture%'
GROUP BY contract_type, part
ORDER BY contract_type
""")

┌────────────────────────┬──────────┬───────┐
│     contract_type      │   part   │   n   │
│        varchar         │ varchar  │ int64 │
├────────────────────────┼──────────┼───────┤
│ Affiliate Agreement    │ Part_III │     1 │
│ Affiliate_Agreements   │ Part_I   │     9 │
│ Endorsement            │ Part_I   │    13 │
│ Endorsement            │ Part_II  │     2 │
│ Endorsement Agreement  │ Part_III │     9 │
│ Joint Venture          │ Part_I   │     9 │
│ Joint Venture _ Filing │ Part_III │    14 │
└────────────────────────┴──────────┴───────┘

In [16]:
con.sql("""
SELECT n_chars, n_chars // 4 AS tokens
FROM contracts
WHERE doc_id LIKE 'LIMEENERGY%'
""")

┌─────────┬────────┐
│ n_chars │ tokens │
│  int32  │ int32  │
├─────────┼────────┤
│   54290 │  13572 │
└─────────┴────────┘

In [17]:
con.sql("""
SELECT
    quantile_cont(n_chars // 4, [0.5, 0.9])::INT[] AS p50_p90,
    max(n_chars // 4) AS max_tokens,
    count(*) FILTER (WHERE n_chars // 4 <= 32000) AS fits_32k,
    count(*) FILTER (WHERE n_chars // 4 > 32000) AS over_32k,
    sum(n_chars // 4) AS tokens_all
FROM contracts
""")

┌───────────────┬────────────┬──────────┬──────────┬────────────┐
│    p50_p90    │ max_tokens │ fits_32k │ over_32k │ tokens_all │
│    int32[]    │   int32    │  int64   │  int64   │   int128   │
├───────────────┼────────────┼──────────┼──────────┼────────────┤
│ [8286, 30529] │      84552 │      461 │       49 │    6701586 │
└───────────────┴────────────┴──────────┴──────────┴────────────┘

In [18]:
con.sql("""
SELECT category, count(*) AS n_rows, bool_or(is_present) AS present
FROM labels
WHERE doc_id LIKE 'LIMEENERGY%'
  AND category IN ('Parties', 'Governing Law', 'Non-Compete')
GROUP BY category
ORDER BY category
""")

┌───────────────┬────────┬─────────┐
│   category    │ n_rows │ present │
│    varchar    │ int64  │ boolean │
├───────────────┼────────┼─────────┤
│ Governing Law │      1 │ true    │
│ Non-Compete   │      1 │ false   │
│ Parties       │      5 │ true    │
└───────────────┴────────┴─────────┘

In [22]:
con.sql("""
WITH per_doc AS (
    SELECT doc_id, category, bool_or(is_present) AS present
    FROM labels
    GROUP BY doc_id, category
)
SELECT category, round(100 * avg(present::INT), 1) AS pct_present
FROM per_doc
GROUP BY category
ORDER BY pct_present DESC
""").show(max_rows=50)

┌────────────────────────────────────┬─────────────┐
│              category              │ pct_present │
│              varchar               │   double    │
├────────────────────────────────────┼─────────────┤
│ Document Name                      │       100.0 │
│ Parties                            │        99.8 │
│ Agreement Date                     │        92.2 │
│ Governing Law                      │        85.7 │
│ Expiration Date                    │        81.0 │
│ Effective Date                     │        76.5 │
│ Anti-Assignment                    │        73.3 │
│ Cap On Liability                   │        53.9 │
│ License Grant                      │        50.0 │
│ Audit Rights                       │        42.0 │
│ Termination For Convenience        │        35.9 │
│ Post-Termination Services          │        35.7 │
│ Exclusivity                        │        35.3 │
│ Renewal Term                       │        34.5 │
│ Insurance                          │        

In [20]:
con.sql("""
SELECT category, count(*) AS n_spans
FROM labels
WHERE doc_id LIKE 'LIMEENERGY%' AND is_present
GROUP BY category
ORDER BY n_spans DESC, category
LIMIT 3
""")

┌────────────────────┬─────────┐
│      category      │ n_spans │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ Warranty Duration  │       8 │
│ Minimum Commitment │       5 │
│ Parties            │       5 │
└────────────────────┴─────────┘

In [21]:
con.sql("""
WITH per_doc AS (
    SELECT doc_id, category, count(*) AS n_spans
    FROM labels
    WHERE is_present
    GROUP BY doc_id, category
)
SELECT category, round(avg(n_spans), 1) AS avg_spans, max(n_spans) AS max_spans
FROM per_doc
GROUP BY category
ORDER BY avg_spans DESC
LIMIT 5
""")


┌────────────────────────────┬───────────┬───────────┐
│          category          │ avg_spans │ max_spans │
│          varchar           │  double   │   int64   │
├────────────────────────────┼───────────┼───────────┤
│ Source Code Escrow         │       5.1 │        11 │
│ Parties                    │       5.0 │        55 │
│ Rofr/Rofo/Rofn             │       4.3 │        22 │
│ Insurance                  │       3.4 │        17 │
│ Affiliate License-Licensor │       3.0 │        10 │
└────────────────────────────┴───────────┴───────────┘

In [25]:
con.sql("""
SELECT span_text, length(span_text) AS n_chars
FROM labels
WHERE category = 'Agreement Date' AND is_present
ORDER BY doc_id
LIMIT 3
""")

┌─────────────────┬─────────┐
│    span_text    │ n_chars │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ June 21, 1999   │      13 │
│ October 1, 2019 │      15 │
│ June 8, 2010    │      12 │
└─────────────────┴─────────┘

In [26]:
con.sql("""
SELECT category,
    quantile_cont(length(span_text), 0.5)::INT AS p50,
    quantile_cont(length(span_text), 0.9)::INT AS p90
FROM labels
WHERE is_present AND category IN (
    'Parties', 'Agreement Date', 'Governing Law',
    'Non-Compete', 'Change Of Control', 'Anti-Assignment',
    'Renewal Term', 'Notice Period To Terminate Renewal', 'Cap On Liability',
    'License Grant', 'Ip Ownership Assignment', 'Non-Transferable License')
GROUP BY category
ORDER BY p50
""")

┌────────────────────────────────────┬───────┬───────┐
│              category              │  p50  │  p90  │
│              varchar               │ int32 │ int32 │
├────────────────────────────────────┼───────┼───────┤
│ Agreement Date                     │    16 │    26 │
│ Parties                            │    17 │    44 │
│ Governing Law                      │   178 │   331 │
│ Anti-Assignment                    │   218 │   585 │
│ Renewal Term                       │   270 │   460 │
│ Notice Period To Terminate Renewal │   298 │   442 │
│ Change Of Control                  │   304 │   772 │
│ Cap On Liability                   │   306 │   705 │
│ Non-Compete                        │   309 │   720 │
│ Ip Ownership Assignment            │   337 │   800 │
│ Non-Transferable License           │   351 │   741 │
│ License Grant                      │   356 │   758 │
└────────────────────────────────────┴───────┴───────┘
  12 rows                                  3 columns

In [27]:
con.sql("""
SELECT DISTINCT lower(split_part(split_part(doc_id, '_', 1), ' - ', 1)) AS family, contract_type
FROM contracts
WHERE doc_id LIKE 'FEDERATED%' OR doc_id LIKE 'Monsanto%'
ORDER BY family
""")

┌────────────────────────────────────────┬───────────────┐
│                 family                 │ contract_type │
│                varchar                 │    varchar    │
├────────────────────────────────────────┼───────────────┤
│ federatedgovernmentincomesecuritiesinc │ Service       │
│ monsanto company                       │ Marketing     │
└────────────────────────────────────────┴───────────────┘

In [28]:
con.sql("""
WITH fam AS (
    SELECT doc_id, lower(split_part(split_part(doc_id, '_', 1), ' - ', 1)) AS family
    FROM contracts
),
sizes AS (
    SELECT family, count(*) AS n_docs FROM fam GROUP BY family
)
SELECT n_docs, count(*) AS n_families
FROM sizes
GROUP BY n_docs
ORDER BY n_docs
""")

┌────────┬────────────┐
│ n_docs │ n_families │
│ int64  │   int64    │
├────────┼────────────┤
│      1 │        431 │
│      2 │         23 │
│      3 │          3 │
│      4 │          6 │
└────────┴────────────┘

In [29]:
con.sql("""
WITH fam AS (
    SELECT doc_id, lower(split_part(split_part(doc_id, '_', 1), ' - ', 1)) AS family
    FROM contracts
)
SELECT family, count(DISTINCT contract_type) AS n_types, count(*) AS n_docs
FROM fam JOIN contracts USING (doc_id)
GROUP BY family
HAVING n_types > 1
ORDER BY n_docs DESC, family
LIMIT 4
""")

┌───────────────────────────────┬─────────┬────────┐
│            family             │ n_types │ n_docs │
│            varchar            │  int64  │ int64  │
├───────────────────────────────┼─────────┼────────┤
│ lejuholdingsltd               │       2 │      3 │
│ alliedesportsentertainmentinc │       2 │      2 │
│ ecosciencesolutionsinc        │       2 │      2 │
│ gpaqacquisitionholdingsinc    │       2 │      2 │
└───────────────────────────────┴─────────┴────────┘

In [35]:
con.sql("SELECT count(*) AS n FROM labels")

┌───────┐
│   n   │
│ int64 │
├───────┤
│ 28031 │
└───────┘